# NVARC batch and prefix-reuse preflight

This Colab experiment validates the real frozen NVARC checkpoint at the exact boundary used by `NvarcEncoder`. It answers two questions:

1. Do real positions produce numerically equivalent hidden states when differently sized source sequences are run alone or together in one right-padded batch?
2. Can a target-conditioning prefix be sliced from one complete causal source forward instead of recomputing the truncated prefix?

The comparison is performed separately for `output.hidden_states[9]`, `[18]`, and `[27]`, and again after concatenating them to width 7680. This is an interface preflight, not a training run and not an ARC quality evaluation.

## Before running

1. Select a Colab GPU runtime with BF16 support and at least 20 GiB VRAM. An A100 is recommended.
2. Confirm that the NVARC checkpoint already exists at `MyDrive/diffusion-arc/models/nvarc-qwen3-4b-grids15-sft139`.
3. Add a Colab Secret named `GITHUB_TOKEN` with read-only access to the private `joshuakauling/diffusion-arc` repository and enable notebook access.
4. Run all cells from top to bottom.

The notebook downloads Python packages and the repository source archive. It does **not** download a model or dataset. The checkpoint loader is forced into local-files-only mode. A timestamped JSON report is written under `MyDrive/diffusion-arc/results/nvarc-batch-preflight`.

## 1. Mount Drive, fix the experiment configuration, and verify the runtime

The tolerances below are declared before any model output is observed. The optional 8192-token capacity probe is disabled by default and is not part of the semantic pass/fail decision.

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
import json
import os
from pathlib import Path
import platform

import torch
from google.colab import drive

SEED = 7
SOURCE_COMMIT = "659ede09ef621a0f174ec420ecfcb4b024af60b3"
NVARC_SOURCE_HANDLE = "sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1"
ATTENTION_IMPLEMENTATION = "sdpa"
SELECTED_HIDDEN_STATE_INDICES = (9, 18, 27)
EXPECTED_VOCAB_SIZE = 16
EXPECTED_PAD_TOKEN_ID = 13
EXPECTED_EOS_TOKEN_ID = 15
EXPECTED_HIDDEN_SIZE = 2560
EXPECTED_LAYER_COUNT = 36
EXPECTED_CONTEXT_WIDTH = 7680
EXPECTED_MAX_TOKENS = 8192
ABSOLUTE_TOLERANCE = 5e-2
RELATIVE_TOLERANCE = 2e-2
RUN_LONG_CONTEXT_STRESS = False
LONG_CONTEXT_TOKENS = 8192
MINIMUM_VRAM_GIB = 20.0

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a GPU runtime in Colab first.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This experiment requires a CUDA GPU with BF16 support.")

gpu = torch.cuda.get_device_properties(0)
gpu_gib = gpu.total_memory / 2**30
SDPA_BACKEND_POLICY = {
    "flash_enabled": torch.backends.cuda.flash_sdp_enabled(),
    "memory_efficient_enabled": torch.backends.cuda.mem_efficient_sdp_enabled(),
    "math_enabled": torch.backends.cuda.math_sdp_enabled(),
}
if gpu_gib < MINIMUM_VRAM_GIB:
    raise RuntimeError(
        f"The selected GPU exposes only {gpu_gib:.1f} GiB; "
        f"this preflight requires at least {MINIMUM_VRAM_GIB:.0f} GiB."
    )

DRIVE_MOUNT = Path("/content/drive")
MY_DRIVE = DRIVE_MOUNT / "MyDrive"
if not MY_DRIVE.is_dir():
    drive.mount(str(DRIVE_MOUNT), force_remount=True)
if not MY_DRIVE.is_dir():
    raise RuntimeError("Google Drive was mounted, but MyDrive is not accessible.")

DEVICE = torch.device("cuda:0")
NVARC_DTYPE = torch.bfloat16
MODEL_ROOT = MY_DRIVE / "diffusion-arc" / "models"
NVARC_PATH = MODEL_ROOT / "nvarc-qwen3-4b-grids15-sft139"
RESULTS_ROOT = MY_DRIVE / "diffusion-arc" / "results" / "nvarc-batch-preflight"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
REPORT_DIR = RESULTS_ROOT / RUN_ID
REPORT_PATH = REPORT_DIR / "report.json"
REPORT_DIR.mkdir(parents=True, exist_ok=False)

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


def save_report(payload: dict[str, object]) -> None:
    REPORT_PATH.write_text(
        json.dumps(payload, indent=2, sort_keys=True, allow_nan=False) + "\n",
        encoding="utf-8",
    )


required_nvarc_files = (
    "added_tokens.json",
    "config.json",
    "generation_config.json",
    "model-00001-of-00002.safetensors",
    "model-00002-of-00002.safetensors",
    "model.safetensors.index.json",
    "special_tokens_map.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "vocab.json",
)
expected_shard_sizes = {
    "model-00001-of-00002.safetensors": 4_996_836_472,
    "model-00002-of-00002.safetensors": 2_270_397_024,
}
missing_nvarc_files = [
    name for name in required_nvarc_files if not (NVARC_PATH / name).is_file()
]
wrong_shard_sizes = {
    name: (expected, (NVARC_PATH / name).stat().st_size)
    for name, expected in expected_shard_sizes.items()
    if (NVARC_PATH / name).is_file()
    and (NVARC_PATH / name).stat().st_size != expected
}
if missing_nvarc_files or wrong_shard_sizes:
    save_report(
        {
            "schema_version": 1,
            "status": "FAIL",
            "stage": "checkpoint-files",
            "checkpoint_path": str(NVARC_PATH),
            "missing_files": missing_nvarc_files,
            "wrong_shard_sizes": wrong_shard_sizes,
        }
    )
    raise FileNotFoundError(
        f"Incomplete or unexpected local NVARC checkpoint; see {REPORT_PATH}."
    )

print(f"Python: {platform.python_version()}")
print(f"Torch: {torch.__version__} | CUDA runtime: {torch.version.cuda}")
print(f"GPU: {gpu.name} | VRAM: {gpu_gib:.1f} GiB | BF16: yes")
print(f"PyTorch SDPA policy: {SDPA_BACKEND_POLICY}")
print(f"Local NVARC checkpoint: {NVARC_PATH}")
print(f"Report: {REPORT_PATH}")
print(f"Predeclared tolerance: atol={ABSOLUTE_TOLERANCE}, rtol={RELATIVE_TOLERANCE}")

Mounted at /content/drive
Python: 3.12.13
Torch: 2.11.0+cu128 | CUDA runtime: 12.8
GPU: NVIDIA A100-SXM4-40GB | VRAM: 39.5 GiB | BF16: yes
PyTorch SDPA policy: {'flash_enabled': True, 'memory_efficient_enabled': True, 'math_enabled': True}
Local NVARC checkpoint: /content/drive/MyDrive/diffusion-arc/models/nvarc-qwen3-4b-grids15-sft139
Report: /content/drive/MyDrive/diffusion-arc/results/nvarc-batch-preflight/20260819T160051Z/report.json
Predeclared tolerance: atol=0.05, rtol=0.02


## 2. Fetch the private repository at the pinned commit

The source commit is immutable and recorded in the report. `GITHUB_TOKEN` is read inside the fetch function and is never written to Drive, placed in a URL, or printed. The archive redirect is accepted only for GitHub's official codeload host.

In [ ]:
import hashlib
import re
import shutil
import stat
import tempfile
import zipfile
from pathlib import PurePosixPath
from urllib.parse import urlparse

import requests
from google.colab import userdata

GITHUB_OWNER = "joshuakauling"
GITHUB_REPOSITORY = "diffusion-arc"
SOURCE_DIR = Path(f"/content/diffusion-arc-{SOURCE_COMMIT[:12]}")
MAX_ARCHIVE_BYTES = 200 * 1024 * 1024


def require_safe_zip(archive_path: Path, destination: Path) -> str:
    if not zipfile.is_zipfile(archive_path):
        raise RuntimeError("GitHub did not return a valid ZIP archive.")
    destination_root = destination.resolve()
    top_levels: set[str] = set()
    with zipfile.ZipFile(archive_path) as archive:
        corrupt_member = archive.testzip()
        if corrupt_member is not None:
            raise RuntimeError("The downloaded source archive is corrupt.")
        for info in archive.infolist():
            member = PurePosixPath(info.filename)
            if member.is_absolute() or ".." in member.parts or not member.parts:
                raise RuntimeError("The source archive contains an unsafe path.")
            if stat.S_ISLNK(info.external_attr >> 16):
                raise RuntimeError("The source archive contains a symbolic link.")
            target = (destination_root / Path(*member.parts)).resolve()
            try:
                target.relative_to(destination_root)
            except ValueError as error:
                raise RuntimeError(
                    "The source archive escapes its extraction directory."
                ) from error
            top_levels.add(member.parts[0])
        if len(top_levels) != 1:
            raise RuntimeError(
                "Expected exactly one top-level directory in the source archive."
            )
        archive.extractall(destination_root)
    return next(iter(top_levels))


def prepare_private_source() -> tuple[Path, str, str]:
    commit_marker = SOURCE_DIR / ".source-commit"
    digest_marker = SOURCE_DIR / ".source-archive-sha256"
    required = SOURCE_DIR / "src" / "diffusion_arc" / "model" / "nvarc.py"
    if SOURCE_DIR.is_dir():
        if (
            required.is_file()
            and commit_marker.is_file()
            and digest_marker.is_file()
            and commit_marker.read_text(encoding="utf-8").strip() == SOURCE_COMMIT
        ):
            archive_sha256 = digest_marker.read_text(encoding="utf-8").strip()
            if re.fullmatch(r"[0-9a-f]{64}", archive_sha256) is None:
                raise RuntimeError("The reused source archive digest is invalid.")
            return SOURCE_DIR, "reused", archive_sha256
        raise RuntimeError(
            f"{SOURCE_DIR} already exists but is incomplete or unverified. "
            "Reconnect the disposable Colab runtime and rerun the notebook."
        )

    if re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT) is None:
        raise RuntimeError("SOURCE_COMMIT must be a full 40-character Git commit SHA.")
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError(
            "Colab Secret GITHUB_TOKEN is missing or notebook access is disabled."
        ) from None
    if not isinstance(token, str) or not token.strip():
        raise RuntimeError("Colab Secret GITHUB_TOKEN is empty.")
    token = token.strip()

    api_root = f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPOSITORY}"
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }
    verification = requests.get(
        f"{api_root}/commits/{SOURCE_COMMIT}", headers=headers, timeout=(10, 60)
    )
    verification_status = verification.status_code
    if verification_status != 200:
        verification.close()
        raise RuntimeError(
            f"GitHub commit verification failed with HTTP {verification_status}. "
            "Check the secret and its repository permissions."
        )
    resolved_commit = verification.json().get("sha")
    verification.close()
    if resolved_commit != SOURCE_COMMIT:
        raise RuntimeError("GitHub did not resolve the exact pinned commit.")

    initial = requests.get(
        f"{api_root}/zipball/{SOURCE_COMMIT}",
        headers=headers,
        stream=True,
        allow_redirects=False,
        timeout=(10, 120),
    )
    del token, headers
    if initial.status_code in {301, 302, 303, 307, 308}:
        location = initial.headers.get("Location", "")
        initial.close()
        parsed = urlparse(location)
        if parsed.scheme != "https" or parsed.hostname != "codeload.github.com":
            raise RuntimeError("GitHub returned an unexpected archive redirect host.")
        response = requests.get(
            location, stream=True, allow_redirects=False, timeout=(10, 120)
        )
        del location
    else:
        response = initial
    if response.status_code != 200:
        status = response.status_code
        response.close()
        raise RuntimeError(f"GitHub source download failed with HTTP {status}.")

    with tempfile.TemporaryDirectory(
        prefix="diffusion-arc-source-", dir="/content"
    ) as temp:
        temp_dir = Path(temp)
        partial_archive = temp_dir / "source.zip.part"
        archive_path = temp_dir / "source.zip"
        digest = hashlib.sha256()
        downloaded = 0
        try:
            with partial_archive.open("wb") as handle:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if not chunk:
                        continue
                    downloaded += len(chunk)
                    if downloaded > MAX_ARCHIVE_BYTES:
                        raise RuntimeError(
                            "The source archive exceeded the 200 MiB safety limit."
                        )
                    digest.update(chunk)
                    handle.write(chunk)
        finally:
            response.close()
        partial_archive.replace(archive_path)

        extraction_dir = temp_dir / "extracted"
        extraction_dir.mkdir()
        top_level = require_safe_zip(archive_path, extraction_dir)
        extracted_source = extraction_dir / top_level
        if not (extracted_source / "pyproject.toml").is_file():
            raise RuntimeError(
                "The archive does not contain the expected Python project."
            )
        shutil.move(str(extracted_source), str(SOURCE_DIR))
        archive_sha256 = digest.hexdigest()
        commit_marker.write_text(SOURCE_COMMIT + "\n", encoding="utf-8")
        digest_marker.write_text(archive_sha256 + "\n", encoding="utf-8")

    return SOURCE_DIR, "downloaded", archive_sha256


SOURCE_DIR, source_status, SOURCE_ARCHIVE_SHA256 = prepare_private_source()
print(f"Private source: {source_status} at commit {SOURCE_COMMIT[:12]}")
print(f"Archive SHA-256: {SOURCE_ARCHIVE_SHA256}")
print(f"Runtime path: {SOURCE_DIR}")

Private source: downloaded at commit 659ede09ef62
Archive SHA-256: 954e49ffcf38867764530416f41cf17c11165e4342cd3d4e4cad82f39921812d
Runtime path: /content/diffusion-arc-659ede09ef62


## 3. Install the minimal pinned dependencies

This deliberately does not install `requirements.txt` and does not replace Colab's CUDA-enabled Torch build. The repository is installed editable without dependency resolution.

In [ ]:
import importlib.metadata
import subprocess
import sys

PINNED_PACKAGES = (
    "accelerate==1.12.0",
    "einops==0.8.1",
    "safetensors==0.8.0",
    "transformers==4.56.1",
)


def run_quiet(label: str, command: list[str]) -> None:
    completed = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
    if completed.returncode != 0:
        tail = "\n".join(completed.stdout.splitlines()[-40:])
        print(tail)
        raise RuntimeError(f"{label} failed with exit code {completed.returncode}.")
    print(f"{label}: ok")


run_quiet(
    "Pinned dependencies",
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        "--no-input",
        *PINNED_PACKAGES,
    ],
)
run_quiet(
    "Diffusion-ARC source",
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        "--no-input",
        "--no-deps",
        "--no-build-isolation",
        "-e",
        str(SOURCE_DIR),
    ],
)

import importlib

source_path = str((SOURCE_DIR / "src").resolve())
if source_path not in sys.path:
    sys.path.insert(0, source_path)

importlib.invalidate_caches()
import diffusion_arc


package_path = Path(diffusion_arc.__file__).resolve()
if SOURCE_DIR.resolve() not in package_path.parents:
    raise RuntimeError(f"diffusion_arc was imported from an unexpected path: {package_path}")
INSTALLED_VERSIONS = {
    distribution: importlib.metadata.version(distribution)
    for distribution in ("accelerate", "einops", "safetensors", "transformers")
}
print(f"diffusion_arc: {package_path}")
print(INSTALLED_VERSIONS)

Pinned dependencies: ok
Diffusion-ARC source: ok
diffusion_arc: /content/diffusion-arc-659ede09ef62/src/diffusion_arc/__init__.py
{'accelerate': '1.12.0', 'einops': '0.8.1', 'safetensors': '0.8.0', 'transformers': '4.56.1'}


## 4. Load the local checkpoint and establish its immutable contract

The model and tokenizer are loaded only from the mounted Drive directory. The preflight fixes BF16 and SDPA explicitly, freezes every parameter, and fails before the numerical experiment if the known 16-token NVARC/Qwen contract is not present.

In [ ]:
import time

from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()

load_started = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(
    NVARC_PATH,
    local_files_only=True,
    trust_remote_code=False,
)
nvarc = AutoModelForCausalLM.from_pretrained(
    NVARC_PATH,
    torch_dtype=NVARC_DTYPE,
    device_map={"": 0},
    local_files_only=True,
    low_cpu_mem_usage=True,
    attn_implementation=ATTENTION_IMPLEMENTATION,
    trust_remote_code=False,
)
nvarc.requires_grad_(False)
nvarc.eval()
load_seconds = time.perf_counter() - load_started

contract_records: list[dict[str, object]] = []


def record_contract(
    name: str,
    observed: object,
    expected: object,
    *,
    passed: bool | None = None,
) -> None:
    contract_records.append(
        {
            "name": name,
            "observed": observed,
            "expected": expected,
            "passed": observed == expected if passed is None else bool(passed),
        }
    )


expected_token_ids = {str(value): value for value in range(10)}
expected_token_ids.update(
    {
        "Ċ": 10,
        "user": 11,
        "assistant": 12,
        "<|endoftext|>": 13,
        "<|im_start|>": 14,
        "<|im_end|>": 15,
    }
)
observed_token_ids = {
    token: tokenizer.convert_tokens_to_ids(token) for token in expected_token_ids
}
canonical_probe = "<|im_start|>user\n0<|im_end|><|im_start|>assistant\n"
observed_probe_ids = tokenizer.encode(canonical_probe, add_special_tokens=False)
expected_probe_ids = [14, 11, 10, 0, 15, 14, 12, 10]
input_embeddings = nvarc.get_input_embeddings()
output_embeddings = nvarc.get_output_embeddings()
parameter_dtypes = sorted({str(parameter.dtype) for parameter in nvarc.parameters()})
parameter_devices = sorted({str(parameter.device) for parameter in nvarc.parameters()})

record_contract("tokenizer_length", len(tokenizer), EXPECTED_VOCAB_SIZE)
record_contract("token_ids", observed_token_ids, expected_token_ids)
record_contract("pad_token_id", tokenizer.pad_token_id, EXPECTED_PAD_TOKEN_ID)
record_contract("eos_token_id", tokenizer.eos_token_id, EXPECTED_EOS_TOKEN_ID)
record_contract("canonical_probe_ids", observed_probe_ids, expected_probe_ids)
record_contract("config_vocab_size", nvarc.config.vocab_size, EXPECTED_VOCAB_SIZE)
record_contract("hidden_size", nvarc.config.hidden_size, EXPECTED_HIDDEN_SIZE)
record_contract("layer_count", nvarc.config.num_hidden_layers, EXPECTED_LAYER_COUNT)
record_contract(
    "max_position_embeddings",
    nvarc.config.max_position_embeddings,
    f">={EXPECTED_MAX_TOKENS}",
    passed=nvarc.config.max_position_embeddings >= EXPECTED_MAX_TOKENS,
)
record_contract(
    "input_embedding_rows", input_embeddings.weight.shape[0], EXPECTED_VOCAB_SIZE
)
record_contract(
    "output_embedding_rows",
    None if output_embeddings is None else output_embeddings.weight.shape[0],
    EXPECTED_VOCAB_SIZE,
)
record_contract(
    "parameter_dtypes", parameter_dtypes, [str(NVARC_DTYPE)]
)
record_contract(
    "parameter_devices",
    parameter_devices,
    "CUDA only",
    passed=bool(parameter_devices)
    and all(device.startswith("cuda") for device in parameter_devices),
)
record_contract("model_eval", not nvarc.training, True)
record_contract(
    "all_parameters_frozen",
    all(not parameter.requires_grad for parameter in nvarc.parameters()),
    True,
)
record_contract(
    "attention_implementation",
    getattr(nvarc.config, "_attn_implementation", None),
    ATTENTION_IMPLEMENTATION,
)

failed_contracts = [record for record in contract_records if not record["passed"]]
for record in contract_records:
    marker = "PASS" if record["passed"] else "FAIL"
    print(f"{marker:4} | {record['name']}: {record['observed']}")
if failed_contracts:
    save_report(
        {
            "schema_version": 1,
            "status": "FAIL",
            "stage": "checkpoint-contract",
            "source_commit": SOURCE_COMMIT,
            "checkpoint": {
                "source_handle": NVARC_SOURCE_HANDLE,
                "local_path": str(NVARC_PATH),
                "contract": contract_records,
            },
            "environment": {
                "python": platform.python_version(),
                "torch": torch.__version__,
                "cuda_runtime": torch.version.cuda,
                "gpu": gpu.name,
                "gpu_vram_gib": gpu_gib,
                "gpu_compute_capability": list(torch.cuda.get_device_capability(0)),
                "pytorch_sdpa_policy": SDPA_BACKEND_POLICY,
                "packages": INSTALLED_VERSIONS,
            },
        }
    )
    raise AssertionError(f"NVARC checkpoint contract failed; see {REPORT_PATH}.")

print(f"NVARC loaded and frozen in {load_seconds:.1f} seconds.")

PASS | tokenizer_length: 16
PASS | token_ids: {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9, 'Ċ': 10, 'user': 11, 'assistant': 12, '<|endoftext|>': 13, '<|im_start|>': 14, '<|im_end|>': 15}
PASS | pad_token_id: 13
PASS | eos_token_id: 15
PASS | canonical_probe_ids: [14, 11, 10, 0, 15, 14, 12, 10]
PASS | config_vocab_size: 16
PASS | hidden_size: 2560
PASS | layer_count: 36
PASS | max_position_embeddings: 262144
PASS | input_embedding_rows: 16
PASS | output_embedding_rows: 16
PASS | parameter_dtypes: ['torch.bfloat16']
PASS | parameter_devices: ['cuda:0']
PASS | model_eval: True
PASS | all_parameters_frozen: True
PASS | attention_implementation: sdpa
NVARC loaded and frozen in 166.2 seconds.


## 5. Build deterministic, valid ARC source sequences

The two synthetic tasks intentionally have different token lengths. Each contains three labelled pairs, yielding two target exposures. The complete source passed to NVARC ends at the final assistant header: earlier target outputs are present, while the final target output is omitted. No ARC dataset is read or downloaded.

In [ ]:
from diffusion_arc.arc_data import (
    ArcAugmentationKey,
    ArcAugmentationSpec,
    ArcAugmentedTrainingSample,
    ArcGeometry,
    ArcPair,
    render_training_prompt,
    tokenize_nvarc_training_sample,
)


def grid(*rows: str) -> tuple[tuple[int, ...], ...]:
    return tuple(tuple(int(value) for value in row) for row in rows)


def make_sample(task_id: str, pairs: tuple[ArcPair, ...]) -> ArcAugmentedTrainingSample:
    return ArcAugmentedTrainingSample(
        key=ArcAugmentationKey(
            task_id=task_id,
            augmentation=ArcAugmentationSpec(
                geometry=ArcGeometry.IDENTITY,
                color_mapping=tuple(range(10)),
                pair_indices=tuple(range(len(pairs))),
            ),
        ),
        pairs=pairs,
    )


source_samples = (
    make_sample(
        "synthetic-a",
        (
            ArcPair(input_grid=grid("12", "34"), output_grid=grid("34", "12")),
            ArcPair(input_grid=grid("560", "789"), output_grid=grid("789", "560")),
            ArcPair(
                input_grid=grid("102", "345", "678"),
                output_grid=grid("678", "345", "102"),
            ),
        ),
    ),
    make_sample(
        "synthetic-b",
        (
            ArcPair(input_grid=grid("1110", "1000"), output_grid=grid("0111", "0001")),
            ArcPair(
                input_grid=grid("22002", "20002", "22222"),
                output_grid=grid("22222", "20002", "22002"),
            ),
            ArcPair(
                input_grid=grid("303030", "033330", "300003", "033330"),
                output_grid=grid("030303", "033330", "300003", "033330"),
            ),
        ),
    ),
)

source_records: list[dict[str, object]] = []
for sample in source_samples:
    rendered = render_training_prompt(sample)
    tokenized = tokenize_nvarc_training_sample(rendered, tokenizer)
    source_end = tokenized.prefix_token_ends[-1]
    source_ids = torch.tensor(
        tokenized.input_ids[:source_end], dtype=torch.int64, device="cpu"
    )
    if len(tokenized.prefix_token_ends) != 2:
        raise AssertionError("Each synthetic source must expose exactly two targets.")
    source_records.append(
        {
            "name": sample.key.task_id,
            "input_ids": source_ids,
            "prefix_ends": tokenized.prefix_token_ends,
            "prompt_sha256": hashlib.sha256(
                rendered.prompt.encode("utf-8")
            ).hexdigest(),
        }
    )

source_lengths = [record["input_ids"].numel() for record in source_records]
if len(set(source_lengths)) != len(source_lengths):
    raise AssertionError("Synthetic source token lengths must differ.")
for record in source_records:
    print(
        f"{record['name']}: {record['input_ids'].numel()} tokens, "
        f"target prefixes={record['prefix_ends']}"
    )

synthetic-a: 58 tokens, target prefixes=(32, 58)
synthetic-b: 102 tokens, target prefixes=(50, 102)


## 6. Define exact-forward and numerical comparison helpers

Every comparison is evaluated in float32, although NVARC outputs remain BF16. Non-finite values, shape changes, or a failed `torch.allclose` check fail that comparison. Max absolute error, mean absolute error, RMSE, and cosine similarity are retained for diagnosis.

In [ ]:
@torch.no_grad()
def encode_exact(input_ids: torch.Tensor) -> tuple[torch.Tensor, ...]:
    batched_ids = input_ids.to(device=DEVICE).unsqueeze(0)
    attention_mask = torch.ones_like(batched_ids, dtype=torch.bool)
    output = nvarc(
        input_ids=batched_ids,
        attention_mask=attention_mask,
        output_hidden_states=True,
        use_cache=False,
        return_dict=True,
    )
    if len(output.hidden_states) <= max(SELECTED_HIDDEN_STATE_INDICES):
        raise RuntimeError(
            f"NVARC returned only {len(output.hidden_states)} hidden states."
        )
    selected = tuple(
        output.hidden_states[index][0].detach().contiguous()
        for index in SELECTED_HIDDEN_STATE_INDICES
    )
    del output
    return selected


def comparison_record(
    *,
    comparison: str,
    source: str,
    representation: str,
    actual: torch.Tensor,
    expected: torch.Tensor,
    prefix_end: int | None = None,
) -> dict[str, object]:
    shape_equal = tuple(actual.shape) == tuple(expected.shape)
    actual_float = actual.float()
    expected_float = expected.float()
    finite = bool(
        torch.isfinite(actual_float).all().item()
        and torch.isfinite(expected_float).all().item()
    )
    if shape_equal and finite:
        difference = (actual_float - expected_float).abs()
        max_absolute_error = difference.max().item()
        mean_absolute_error = difference.mean().item()
        rmse = torch.sqrt(torch.mean(torch.square(difference))).item()
        cosine_similarity = torch.nn.functional.cosine_similarity(
            actual_float.reshape(-1), expected_float.reshape(-1), dim=0
        ).item()
        close = bool(
            torch.allclose(
                actual_float,
                expected_float,
                atol=ABSOLUTE_TOLERANCE,
                rtol=RELATIVE_TOLERANCE,
            )
        )
    else:
        max_absolute_error = None
        mean_absolute_error = None
        rmse = None
        cosine_similarity = None
        close = False
    return {
        "comparison": comparison,
        "source": source,
        "prefix_end": prefix_end,
        "representation": representation,
        "shape": list(actual.shape),
        "expected_shape": list(expected.shape),
        "dtype": str(actual.dtype),
        "finite": finite,
        "max_absolute_error": max_absolute_error,
        "mean_absolute_error": mean_absolute_error,
        "rmse": rmse,
        "cosine_similarity": cosine_similarity,
        "passed": shape_equal and finite and close,
    }


def compare_selected_states(
    *,
    comparison: str,
    source: str,
    actual: tuple[torch.Tensor, ...],
    expected: tuple[torch.Tensor, ...],
    prefix_end: int | None = None,
) -> list[dict[str, object]]:
    records = [
        comparison_record(
            comparison=comparison,
            source=source,
            prefix_end=prefix_end,
            representation=f"hidden_states[{layer_index}]",
            actual=actual_state,
            expected=expected_state,
        )
        for layer_index, actual_state, expected_state in zip(
            SELECTED_HIDDEN_STATE_INDICES, actual, expected, strict=True
        )
    ]
    records.append(
        comparison_record(
            comparison=comparison,
            source=source,
            prefix_end=prefix_end,
            representation="concatenated_7680",
            actual=torch.cat(actual, dim=-1),
            expected=torch.cat(expected, dim=-1),
        )
    )
    return records

## 7. Run each complete source alone

These unpadded single-sequence forwards are the reference values for the right-padded batch comparison and the source of the prefix slices.

In [ ]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
reference_started = time.perf_counter()
single_references: dict[str, tuple[torch.Tensor, ...]] = {}
reference_summaries: list[dict[str, object]] = []
for record in source_records:
    source_name = record["name"]
    selected_states = encode_exact(record["input_ids"])
    single_references[source_name] = selected_states
    reference_summaries.append(
        {
            "source": source_name,
            "shapes": [list(state.shape) for state in selected_states],
            "dtypes": [str(state.dtype) for state in selected_states],
            "finite": all(torch.isfinite(state).all().item() for state in selected_states),
        }
    )
reference_seconds = time.perf_counter() - reference_started
reference_peak_gib = torch.cuda.max_memory_allocated() / 2**30
print(reference_summaries)
print(f"Reference forwards: {reference_seconds:.2f}s, peak allocated {reference_peak_gib:.2f} GiB")

[{'source': 'synthetic-a', 'shapes': [[58, 2560], [58, 2560], [58, 2560]], 'dtypes': ['torch.bfloat16', 'torch.bfloat16', 'torch.bfloat16'], 'finite': True}, {'source': 'synthetic-b', 'shapes': [[102, 2560], [102, 2560], [102, 2560]], 'dtypes': ['torch.bfloat16', 'torch.bfloat16', 'torch.bfloat16'], 'finite': True}]
Reference forwards: 0.91s, peak allocated 6.80 GiB


## 8. Run the same complete sources through `NvarcEncoder`

The runner receives only flat source IDs and source offsets. It reconstructs rows, right-pads them internally, creates the attention mask, runs one NVARC forward with physical batch size two, concatenates the selected states, and removes padded positions.

In [ ]:
from diffusion_arc.model import (
    NVARC_CONTEXT_WIDTH,
    NVARC_HIDDEN_SIZE,
    NVARC_HIDDEN_STATE_INDICES,
    NVARC_PAD_TOKEN_ID,
    NvarcEncoder,
)

flat_input_ids = torch.cat([record["input_ids"] for record in source_records])
offset_values = [0]
for record in source_records:
    offset_values.append(offset_values[-1] + record["input_ids"].numel())
sequence_offsets = torch.tensor(offset_values, dtype=torch.int64, device="cpu")
runner = NvarcEncoder(nvarc)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
batch_started = time.perf_counter()
packed_hidden_states = runner(
    input_ids=flat_input_ids,
    sequence_offsets=sequence_offsets,
)
torch.cuda.synchronize()
batch_seconds = time.perf_counter() - batch_started
batch_peak_gib = torch.cuda.max_memory_allocated() / 2**30

runner_contract_checks = {
    "shape_T_by_7680": tuple(packed_hidden_states.hidden_states.shape)
    == (flat_input_ids.numel(), EXPECTED_CONTEXT_WIDTH),
    "offset_tensor_identity": packed_hidden_states.sequence_offsets is sequence_offsets,
    "offset_values_unchanged": torch.equal(
        packed_hidden_states.sequence_offsets, sequence_offsets
    ),
    "bf16_output": packed_hidden_states.hidden_states.dtype == NVARC_DTYPE,
    "cuda_output": packed_hidden_states.hidden_states.device.type == "cuda",
    "no_gradient_tracking": not packed_hidden_states.hidden_states.requires_grad
    and packed_hidden_states.hidden_states.grad_fn is None,
    "finite_output": bool(
        torch.isfinite(packed_hidden_states.hidden_states).all().item()
    ),
    "runner_eval": not runner.training and not runner.model.training,
    "all_parameters_frozen": all(
        not parameter.requires_grad for parameter in runner.model.parameters()
    ),
    "fixed_pad_token_id": NVARC_PAD_TOKEN_ID == EXPECTED_PAD_TOKEN_ID,
    "fixed_hidden_size": NVARC_HIDDEN_SIZE == EXPECTED_HIDDEN_SIZE,
    "fixed_selected_layers": tuple(NVARC_HIDDEN_STATE_INDICES)
    == SELECTED_HIDDEN_STATE_INDICES,
    "fixed_context_width": NVARC_CONTEXT_WIDTH == EXPECTED_CONTEXT_WIDTH,
}

batch_comparisons: list[dict[str, object]] = []
for source_index, record in enumerate(source_records):
    start = offset_values[source_index]
    end = offset_values[source_index + 1]
    packed_source = packed_hidden_states.hidden_states[start:end]
    packed_layers = tuple(
        packed_source[:, layer_offset : layer_offset + EXPECTED_HIDDEN_SIZE]
        for layer_offset in range(0, EXPECTED_CONTEXT_WIDTH, EXPECTED_HIDDEN_SIZE)
    )
    batch_comparisons.extend(
        compare_selected_states(
            comparison="single_vs_right_padded_batch",
            source=record["name"],
            actual=packed_layers,
            expected=single_references[record["name"]],
        )
    )

print(runner_contract_checks)
print(f"Batched runner: {batch_seconds:.2f}s, peak allocated {batch_peak_gib:.2f} GiB")
for result in batch_comparisons:
    print(
        f"{'PASS' if result['passed'] else 'FAIL'} | {result['source']} | "
        f"{result['representation']} | max_abs={result['max_absolute_error']}"
    )

{'shape_T_by_7680': True, 'offset_tensor_identity': True, 'offset_values_unchanged': True, 'bf16_output': True, 'cuda_output': True, 'no_gradient_tracking': True, 'finite_output': True, 'runner_eval': True, 'all_parameters_frozen': True, 'fixed_pad_token_id': True, 'fixed_hidden_size': True, 'fixed_selected_layers': True, 'fixed_context_width': True}
Batched runner: 0.19s, peak allocated 6.83 GiB
FAIL | synthetic-a | hidden_states[9] | max_abs=0.5
FAIL | synthetic-a | hidden_states[18] | max_abs=0.5
FAIL | synthetic-a | hidden_states[27] | max_abs=1.0
FAIL | synthetic-a | concatenated_7680 | max_abs=1.0
FAIL | synthetic-b | hidden_states[9] | max_abs=192.0
FAIL | synthetic-b | hidden_states[18] | max_abs=192.0
FAIL | synthetic-b | hidden_states[27] | max_abs=192.0
FAIL | synthetic-b | concatenated_7680 | max_abs=192.0


## 9. Validate causal prefix reuse for every target exposure

For each recorded `prefix_end`, the cached slice from the complete source forward is compared with a new NVARC forward that receives only `input_ids[:prefix_end]`. This directly tests the assumption needed to derive all target-conditioning prefixes after one complete-source NVARC pass.

In [ ]:
prefix_started = time.perf_counter()
prefix_comparisons: list[dict[str, object]] = []
for record in source_records:
    source_name = record["name"]
    full_source_states = single_references[source_name]
    for prefix_end in record["prefix_ends"]:
        truncated_states = encode_exact(record["input_ids"][:prefix_end])
        sliced_states = tuple(
            full_state[:prefix_end] for full_state in full_source_states
        )
        prefix_comparisons.extend(
            compare_selected_states(
                comparison="truncated_forward_vs_full_source_slice",
                source=source_name,
                prefix_end=prefix_end,
                actual=truncated_states,
                expected=sliced_states,
            )
        )
        del truncated_states, sliced_states
prefix_seconds = time.perf_counter() - prefix_started

for result in prefix_comparisons:
    print(
        f"{'PASS' if result['passed'] else 'FAIL'} | {result['source']} | "
        f"prefix={result['prefix_end']} | {result['representation']} | "
        f"max_abs={result['max_absolute_error']}"
    )
print(f"Truncated-prefix forwards: {prefix_seconds:.2f}s")

FAIL | synthetic-a | prefix=32 | hidden_states[9] | max_abs=16.0
FAIL | synthetic-a | prefix=32 | hidden_states[18] | max_abs=16.0
FAIL | synthetic-a | prefix=32 | hidden_states[27] | max_abs=16.0
FAIL | synthetic-a | prefix=32 | concatenated_7680 | max_abs=16.0
PASS | synthetic-a | prefix=58 | hidden_states[9] | max_abs=0.0
PASS | synthetic-a | prefix=58 | hidden_states[18] | max_abs=0.0
PASS | synthetic-a | prefix=58 | hidden_states[27] | max_abs=0.0
PASS | synthetic-a | prefix=58 | concatenated_7680 | max_abs=0.0
FAIL | synthetic-b | prefix=50 | hidden_states[9] | max_abs=192.0
FAIL | synthetic-b | prefix=50 | hidden_states[18] | max_abs=192.0
FAIL | synthetic-b | prefix=50 | hidden_states[27] | max_abs=192.0
FAIL | synthetic-b | prefix=50 | concatenated_7680 | max_abs=192.0
PASS | synthetic-b | prefix=102 | hidden_states[9] | max_abs=0.0
PASS | synthetic-b | prefix=102 | hidden_states[18] | max_abs=0.0
PASS | synthetic-b | prefix=102 | hidden_states[27] | max_abs=0.0
PASS | synthet

## 10. Optional 8192-token capacity probe

Set `RUN_LONG_CONTEXT_STRESS = True` in the first code cell to run this. It repeats valid vocabulary IDs to test only the configured context capacity, output shapes, and finiteness. It is intentionally excluded from the causal-equivalence verdict because the repeated sequence is not an ARC prompt and the memory requirement is substantially larger.

In [ ]:
long_context_stress: dict[str, object] = {"enabled": RUN_LONG_CONTEXT_STRESS}
if RUN_LONG_CONTEXT_STRESS:
    pattern = source_records[0]["input_ids"]
    repeats = (LONG_CONTEXT_TOKENS + pattern.numel() - 1) // pattern.numel()
    long_ids = pattern.repeat(repeats)[:LONG_CONTEXT_TOKENS]
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    stress_started = time.perf_counter()
    long_states = encode_exact(long_ids)
    torch.cuda.synchronize()
    stress_seconds = time.perf_counter() - stress_started
    long_context_stress.update(
        {
            "token_count": long_ids.numel(),
            "shapes": [list(state.shape) for state in long_states],
            "dtypes": [str(state.dtype) for state in long_states],
            "finite": all(torch.isfinite(state).all().item() for state in long_states),
            "seconds": stress_seconds,
            "peak_allocated_gib": torch.cuda.max_memory_allocated() / 2**30,
        }
    )
    del long_ids, long_states
else:
    long_context_stress["reason"] = "Disabled; not part of semantic pass/fail."
print(long_context_stress)

{'enabled': False, 'reason': 'Disabled; not part of semantic pass/fail.'}


## 11. Save the machine-readable report

The report is written before the final assertion. Therefore numerical or runner-contract failures remain available in Drive even when the last cell raises. No checkpoint tensors, prompt text, model weights, or secret values are serialized.

In [ ]:
all_comparisons = batch_comparisons + prefix_comparisons
failed_comparisons = [result for result in all_comparisons if not result["passed"]]
failed_runner_checks = [
    name for name, passed in runner_contract_checks.items() if not passed
]
experiment_passed = not failed_contracts and not failed_runner_checks and not failed_comparisons

source_summaries = [
    {
        "name": record["name"],
        "token_count": record["input_ids"].numel(),
        "prefix_ends": list(record["prefix_ends"]),
        "target_count": len(record["prefix_ends"]),
        "prompt_sha256": record["prompt_sha256"],
    }
    for record in source_records
]
report = {
    "schema_version": 1,
    "status": "PASS" if experiment_passed else "FAIL",
    "run_id": RUN_ID,
    "question": (
        "Whether right-padded NVARC batching preserves real-position hidden states "
        "and whether complete-source hidden states can be sliced for causal target prefixes."
    ),
    "source": {
        "repository": f"{GITHUB_OWNER}/{GITHUB_REPOSITORY}",
        "commit": SOURCE_COMMIT,
        "archive_sha256": SOURCE_ARCHIVE_SHA256,
    },
    "checkpoint": {
        "source_handle": NVARC_SOURCE_HANDLE,
        "local_path": str(NVARC_PATH),
        "contract": contract_records,
        "parameter_count": sum(parameter.numel() for parameter in nvarc.parameters()),
    },
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda_runtime": torch.version.cuda,
        "gpu": gpu.name,
        "gpu_vram_gib": gpu_gib,
        "gpu_compute_capability": list(torch.cuda.get_device_capability(0)),
        "bf16_supported": torch.cuda.is_bf16_supported(),
        "attention_implementation": getattr(
            nvarc.config, "_attn_implementation", None
        ),
        "pytorch_sdpa_policy": SDPA_BACKEND_POLICY,
        "packages": INSTALLED_VERSIONS,
    },
    "configuration": {
        "seed": SEED,
        "dtype": str(NVARC_DTYPE),
        "selected_hidden_state_indices": list(SELECTED_HIDDEN_STATE_INDICES),
        "absolute_tolerance": ABSOLUTE_TOLERANCE,
        "relative_tolerance": RELATIVE_TOLERANCE,
        "long_context_stress_enabled": RUN_LONG_CONTEXT_STRESS,
    },
    "sources": source_summaries,
    "single_reference_outputs": reference_summaries,
    "runner": {
        "physical_batch_size": len(source_records),
        "sequence_offsets": sequence_offsets.tolist(),
        "output_shape": list(packed_hidden_states.hidden_states.shape),
        "output_dtype": str(packed_hidden_states.hidden_states.dtype),
        "output_device": str(packed_hidden_states.hidden_states.device),
        "contract_checks": runner_contract_checks,
    },
    "comparisons": all_comparisons,
    "timings_seconds": {
        "model_load": load_seconds,
        "single_references": reference_seconds,
        "right_padded_batch": batch_seconds,
        "truncated_prefixes": prefix_seconds,
    },
    "peak_allocated_cuda_memory_gib": {
        "single_references": reference_peak_gib,
        "right_padded_batch": batch_peak_gib,
    },
    "long_context_stress": long_context_stress,
    "failure_summary": {
        "runner_contract_checks": failed_runner_checks,
        "comparison_count": len(failed_comparisons),
    },
}
save_report(report)

print(f"Result: {report['status']}")
print(
    f"Comparisons: {len(all_comparisons) - len(failed_comparisons)}/"
    f"{len(all_comparisons)} passed"
)
print(f"Saved report: {REPORT_PATH}")

Result: FAIL
Comparisons: 8/24 passed
Saved report: /content/drive/MyDrive/diffusion-arc/results/nvarc-batch-preflight/20260819T160051Z/report.json


In [ ]:
if not experiment_passed:
    failed_labels = [
        (
            result["comparison"],
            result["source"],
            result["prefix_end"],
            result["representation"],
        )
        for result in failed_comparisons
    ]
    raise AssertionError(
        f"NVARC preflight failed. Runner checks={failed_runner_checks}; "
        f"comparisons={failed_labels}. Full report: {REPORT_PATH}"
    )
print(
    "PASS: right-padded batching and complete-source prefix reuse satisfy the "
    "declared tolerance."
)

AssertionError: NVARC preflight failed. Runner checks=[]; comparisons=[('single_vs_right_padded_batch', 'synthetic-a', None, 'hidden_states[9]'), ('single_vs_right_padded_batch', 'synthetic-a', None, 'hidden_states[18]'), ('single_vs_right_padded_batch', 'synthetic-a', None, 'hidden_states[27]'), ('single_vs_right_padded_batch', 'synthetic-a', None, 'concatenated_7680'), ('single_vs_right_padded_batch', 'synthetic-b', None, 'hidden_states[9]'), ('single_vs_right_padded_batch', 'synthetic-b', None, 'hidden_states[18]'), ('single_vs_right_padded_batch', 'synthetic-b', None, 'hidden_states[27]'), ('single_vs_right_padded_batch', 'synthetic-b', None, 'concatenated_7680'), ('truncated_forward_vs_full_source_slice', 'synthetic-a', 32, 'hidden_states[9]'), ('truncated_forward_vs_full_source_slice', 'synthetic-a', 32, 'hidden_states[18]'), ('truncated_forward_vs_full_source_slice', 'synthetic-a', 32, 'hidden_states[27]'), ('truncated_forward_vs_full_source_slice', 'synthetic-a', 32, 'concatena

## Interpretation

A PASS establishes the tested checkpoint/runtime contract for these sequences: the runner's padding mask preserved real-token states, and causal prefix slices agreed with separately executed truncated prefixes within the declared BF16 tolerance. It does not prove agreement for a different checkpoint, Transformers version, attention backend, precision, GPU family, or arbitrarily long prompt. Re-run and retain this report whenever one of those execution variables changes.

A FAIL should block connecting prefix reuse to FLUX. Inspect the per-layer errors in `report.json` before changing tolerances or execution policy.